# Mission 02: A Governed Taxi Feed

A partner sends the city's taxi trips through Kafka once a day. The data platform team loads each day into an
Iceberg table that dashboards read. Last quarter, bad deliveries reached the dashboards three times. Build a
loader that **cannot publish bad data**: a schema contract at the producer, **write-audit-publish** on an Iceberg
branch, quality and volume checks, and lineage events for every run.

**Lessons used:** 05 (Iceberg REST catalog, branches), 06 (Avro, Schema Registry, data contracts),
07 (observability, OpenLineage, Marquez).

**Data:** NYC yellow taxi trips 2024-01-08 to 2024-01-17, a 10 % sample (every 10th trip).
Four incidents are built into the deliveries; your loader must find them without knowing them.

**Services:** `docker compose -f docker/compose.yml --profile streaming --profile lakehouse --profile lineage up -d
kafka schema-registry rustfs iceberg-rest postgres marquez`

In [ ]:
import datetime as dt
import json
import struct
import subprocess
import sys
import time
import uuid
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import requests
from confluent_kafka import Consumer, Producer
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroDeserializer, AvroSerializer
from confluent_kafka.serialization import MessageField, SerializationContext
from openlineage.client import OpenLineageClient
from openlineage.client.event_v2 import InputDataset, Job, OutputDataset, Run, RunEvent, RunState
from openlineage.client.facet_v2 import error_message_run, output_statistics_output_dataset
from openlineage.client.transport.http import HttpConfig, HttpTransport
from pyiceberg.catalog import load_catalog

sys.path.insert(0, str(Path.cwd().parent.parent / "tools"))
import datasets                                   # course helper: pinned datasets and Docker services

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})
for profile, svc in [("streaming", "kafka"), ("streaming", "schema-registry"), ("lakehouse", "rustfs"),
                     ("lakehouse", "iceberg-rest"), ("lineage", "postgres")]:
    datasets.ensure_service(profile, svc, timeout=240)
# Marquez's default config expects database, user and password "marquez"
psql = ["docker", "compose", "-f", str(datasets.COMPOSE), "exec", "-T", "postgres", "psql", "-U", "course", "-d", "course", "-tAc"]
if subprocess.run(psql + ["SELECT 1 FROM pg_roles WHERE rolname = 'marquez'"], capture_output=True, text=True).stdout.strip() != "1":
    subprocess.run(psql + ["CREATE ROLE marquez LOGIN PASSWORD 'marquez'"], check=True, capture_output=True)
if subprocess.run(psql + ["SELECT 1 FROM pg_database WHERE datname = 'marquez'"], capture_output=True, text=True).stdout.strip() != "1":
    subprocess.run(psql + ["CREATE DATABASE marquez OWNER marquez"], check=True, capture_output=True)
# each Marquez event is one transaction; on Docker Desktop disks a synchronous commit (fsync) takes seconds
subprocess.run(psql + ["ALTER ROLE marquez SET synchronous_commit = off"], check=True, capture_output=True)
datasets.ensure_service("lineage", "marquez")
for url in ["http://localhost:8181/v1/config", "http://localhost:5001/healthcheck"]:     # REST catalog, Marquez
    for _ in range(90):
        try:
            requests.get(url, timeout=2).raise_for_status(); break
        except requests.RequestException:
            time.sleep(2)

REGISTRY, MARQUEZ, TOPIC = "http://localhost:8081", "http://localhost:5000", "m02-trips"
sr = SchemaRegistryClient({"url": REGISTRY})
admin = AdminClient({"bootstrap.servers": "localhost:9092"})
s3 = boto3.client("s3", region_name="us-east-1", endpoint_url="http://localhost:9000",
                  aws_access_key_id="course", aws_secret_access_key="course-password")
if "lakehouse" not in [b["Name"] for b in s3.list_buckets()["Buckets"]]:
    s3.create_bucket(Bucket="lakehouse")
catalog = load_catalog("lake", **{"type": "rest", "uri": "http://localhost:8181", "s3.endpoint": "http://localhost:9000",
                                  "s3.access-key-id": "course", "s3.secret-access-key": "course-password",
                                  "s3.region": "us-east-1", "s3.path-style-access": "true"})
# clean start: topic, subject and table of earlier runs
if TOPIC in admin.list_topics(timeout=10).topics:
    admin.delete_topics([TOPIC])[TOPIC].result(); time.sleep(3)
admin.create_topics([NewTopic(TOPIC, 1, 1)])[TOPIC].result()
if f"{TOPIC}-value" in sr.get_subjects():
    sr.delete_subject(f"{TOPIC}-value"); sr.delete_subject(f"{TOPIC}-value", permanent=True)
catalog.create_namespace_if_not_exists("m02")
if catalog.table_exists("m02.trips"):
    catalog.purge_table("m02.trips")
print("services ready")

## The Partner's Deliveries (given, do not change)

The cell below plays the partner. It produces one batch per day (message key = day) with Avro and the Schema
Registry. Read it only to understand the message format; your loader must not use its knowledge of the incidents.

In [ ]:
TRIP_V1 = {"type": "record", "name": "Trip", "namespace": "nyc.taxi", "fields": [
    {"name": "trip_id", "type": "long"},
    {"name": "pickup_at", "type": {"type": "long", "logicalType": "timestamp-millis"}},
    {"name": "pu_zone", "type": "int"}, {"name": "do_zone", "type": "int"},
    {"name": "distance_miles", "type": "double"}, {"name": "total_usd", "type": "double"}]}
TRIP_V2 = {**TRIP_V1, "fields": TRIP_V1["fields"] + [{"name": "payment_type", "type": ["null", "int"], "default": None}]}
TRIP_BAD = {**TRIP_V2, "fields": [f if f["name"] != "total_usd" else {"name": "total_usd", "type": "string"}
                                   for f in TRIP_V2["fields"]]}
sr.set_compatibility(subject_name=f"{TOPIC}-value", level="BACKWARD")

src = pd.read_parquet(datasets.taxi("2024-01"), columns=["tpep_pickup_datetime", "PULocationID", "DOLocationID",
                                                         "trip_distance", "total_amount", "payment_type"])
src = src[(src.tpep_pickup_datetime >= "2024-01-08") & (src.tpep_pickup_datetime < "2024-01-18")]
src = src.sort_values("tpep_pickup_datetime").iloc[::10].reset_index(drop=True)
src["trip_id"] = src.index
producer = Producer({"bootstrap.servers": "localhost:9092", "linger.ms": 50})
delivery_log = []
for day, part in src.groupby(src.tpep_pickup_datetime.dt.date):
    schema = TRIP_V1 if day < dt.date(2024, 1, 10) else TRIP_V2
    if day == dt.date(2024, 1, 12):
        schema = TRIP_BAD                                           # an incompatible change
    if day == dt.date(2024, 1, 14):
        part = part.copy(); bad = part.sample(frac=0.3, random_state=1).index
        part.loc[bad, "total_amount"] = -part.loc[bad, "total_amount"].abs()   # a sign bug in 30 % of the rows
    if day == dt.date(2024, 1, 16):
        part = part.iloc[: len(part) // 2]                          # a truncated export
    ser = AvroSerializer(sr, json.dumps(schema), conf={"auto.register.schemas": True})
    ctx = SerializationContext(TOPIC, MessageField.VALUE)
    try:
        for r in part.itertuples():
            rec = {"trip_id": int(r.trip_id), "pickup_at": r.tpep_pickup_datetime.tz_localize("UTC").to_pydatetime(),
                   "pu_zone": int(r.PULocationID), "do_zone": int(r.DOLocationID),
                   "distance_miles": float(r.trip_distance), "total_usd": float(r.total_amount)}
            if schema is TRIP_V2:
                rec["payment_type"] = None if pd.isna(r.payment_type) else int(r.payment_type)
            if schema is TRIP_BAD:
                rec["payment_type"], rec["total_usd"] = None, f"{r.total_amount:.2f}"
            producer.produce(TOPIC, key=str(day), value=ser(rec, ctx))
            producer.poll(0)
        delivery_log.append({"day": day, "partner": "sent", "rows": len(part)})
    except Exception as e:                                          # the registry refused the schema
        delivery_log.append({"day": day, "partner": f"blocked: {type(e).__name__}", "rows": 0})
producer.flush()
print(pd.DataFrame(delivery_log).to_string(index=False))

## Tasks

1. **Consume.** Read the topic, deserialize with the registry, and group the messages by day (the key). Record the
   schema ID of every message (bytes 1–4 after the magic byte).
2. **Contract.** Write the contract as code: required fields, `pu_zone` and `do_zone` in 1–265 (every row), at most
   1 % of `distance_miles` outside 0–200 and at most 5 % of `total_usd` outside 0–1000 (real feeds contain GPS
   errors and refunds). A check returns the list of violated rules.
3. **Write-audit-publish.** Load the first day into `m02.trips` (REST catalog) as the baseline. For every later day:
   create a branch `audit` from `main`, append the batch to the branch, run the contract and a **volume check**
   (rows vs. the median of the last 7 published days, more than 35 % deviation fails), and fast-forward `main` only if
   all checks pass. A day without data is an incident too. When the producer adds an optional field, evolve the table
   schema.
4. **Lineage.** For every daily run, send OpenLineage events to Marquez: START, then COMPLETE (with the row count)
   or FAIL (with the reasons as the error message). Input: the Kafka topic; output: the Iceberg table.
5. **Report.** A table and a chart per day: delivered rows, schema ID, volume deviation, violated rules, published
   or not. Prove that `main` contains exactly the published days, and show the failed runs from Marquez.
6. **Explain.** Five sentences for the dashboard owners: which incidents were stopped, by which control, and why
   no single control would have caught all of them.

In [ ]:
# Tasks 1-6
...

## Submission Checklist

- [ ] Every day 2024-01-08 to 2024-01-17 has a decision: published, or blocked with a reason.
- [ ] Bad data never reaches `main` (prove it with row counts per day on `main`).
- [ ] The table schema evolves when the producer adds `payment_type`; older rows read it as NULL.
- [ ] Every run has lineage events in Marquez; failed runs carry the reason.
- [ ] The explanation names one control per incident.

A solution is in `02-Governed-Taxi-Feed-solution.ipynb`. Try the mission first.